# Notebook 4.0 — Model Training, Evaluation & SHAP Interpretation
---

## Modeling Workflow Summary

This notebook orchestrates the full ML lifecycle for credit-risk scoring.
All heavy logic is encapsulated in `src/train.py` (Backend Engine).
This notebook is a pure **Execution Harness**.

### Workflow Diagram

```
[RAW PARQUET DATA]
       |
       v
[load_data()] --------> X_train, y_train, X_test (OOT), y_test
       |
       v
[compute_scale_pos_weight()] --> scale_pos_weight (XGBoost penalty)
       |
       +----> [rf_optuna_objective()] x 50 trials (TimeSeriesSplit CV, PR-AUC)
       |              |
       |              v
       |       [train_final_model("rf")] --> RF Baseline (full train fit)
       |
       +----> [xgb_optuna_objective()] x 50 trials (TimeSeriesSplit CV, PR-AUC)
                      |
                      v
               [train_final_model("xgb")] --> XGBoost Champion (full train fit)
                      |
                      v
             [evaluate_model()] x2  --> OOT Metrics (ROC-AUC, PR-AUC, F1, Recall)
                      |
             +---------+----------+
             |                    |
             v                    v
   [plot_evaluation_curves()]  [plot_shap_summary()]
   reports/figures/             reports/figures/
   model_evaluation_curves.png  shap_summary_plot.png
             |
             v
   [build_comparison_matrix()] --> reports/model_comparison_matrix.csv
             |
             v
   [print_champion_justification()]
```

In [ ]:
# CELL 1 — Environment Setup

import sys
import logging
from pathlib import Path
import functools

# -- Path setup: add project root to sys.path --
PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# -- Configure root logger to stdout with ASCII-safe format --
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)

# -- Import backend engine --
from src.train import (
    load_data,
    compute_scale_pos_weight,
    rf_optuna_objective,
    xgb_optuna_objective,
    run_optuna_study,
    train_final_model,
    evaluate_model,
    plot_evaluation_curves,
    plot_shap_summary,
    build_comparison_matrix,
    save_models,
    print_champion_justification,
)

print("=" * 60)
print("ENVIRONMENT READY — All modules imported successfully.")
print("=" * 60)

/Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ENVIRONMENT READY — All modules imported successfully.


In [ ]:
# CELL 2 — Path Configuration (single source of truth)

DATA_DIR    = PROJECT_ROOT / "data" / "processed"
MODELS_DIR  = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

TRAIN_PATH  = DATA_DIR / "train_features.parquet"
TEST_PATH   = DATA_DIR / "test_features.parquet"
RF_PATH     = MODELS_DIR / "random_forest_baseline.joblib"
XGB_PATH    = MODELS_DIR / "xgboost_champion.joblib"
CURVES_PATH = FIGURES_DIR / "model_evaluation_curves.png"
SHAP_PATH   = FIGURES_DIR / "shap_summary_plot.png"
MATRIX_PATH = REPORTS_DIR / "model_comparison_matrix.csv"

# -- Create directories if they do not exist --
for d in [MODELS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths configured:")
for name, p in [
    ("Train", TRAIN_PATH), ("Test (OOT)", TEST_PATH),
    ("RF model", RF_PATH), ("XGB model", XGB_PATH),
    ("Eval curves", CURVES_PATH), ("SHAP plot", SHAP_PATH),
    ("Comparison CSV", MATRIX_PATH),
]:
    print(f"  {name:<20} -> {p}")

Paths configured:
  Train                -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/train_features.parquet
  Test (OOT)           -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/test_features.parquet
  RF model             -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/random_forest_baseline.joblib
  XGB model            -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/xgboost_champion.joblib
  Eval curves          -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/model_evaluation_curves.png
  SHAP plot            -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/shap_summary_plot.png
  Comparison CSV       -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/model_comparison_matrix.csv


In [ ]:
# CELL 3 — Load Data

X_train, y_train, X_test, y_test = load_data(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    target_col="target",
)

[DATA LOAD]
+-----------------------+------------------+
| Split                 | Shape            |
+-----------------------+------------------+
| Train features        | (402754, 52)     |
| Train labels          | (402754,)        |
| Test features (OOT)   | (40207, 52)      |
| Test labels  (OOT)    | (40207,)         |
+-----------------------+------------------+
| Train default rate    | 38.58%          |
| Test  default rate    | 26.86%          |
+-----------------------+------------------+


In [ ]:
# CELL 4 — Optuna Hyperparameter Tuning
# N_TRIALS = 10 per model (adjust for speed vs quality)

import optuna

N_TRIALS = 10

scale_pos_weight = compute_scale_pos_weight(y_train)

# -- Random Forest Study --
rf_objective = functools.partial(
    rf_optuna_objective,
    X_train=X_train,
    y_train=y_train,
    n_splits=3,
)
rf_study = run_optuna_study(
    model_name="Random Forest",
    objective_fn=rf_objective,
    n_trials=N_TRIALS,
)

# -- XGBoost Study --
xgb_objective = functools.partial(
    xgb_optuna_objective,
    X_train=X_train,
    y_train=y_train,
    scale_pos_weight=scale_pos_weight,
    n_splits=3,
)
xgb_study = run_optuna_study(
    model_name="XGBoost",
    objective_fn=xgb_objective,
    n_trials=N_TRIALS,
)

[CLASS WEIGHT] Computed scale_pos_weight: 1.5917
[OPTUNA] Running 10 trials for Random Forest ...
[OPTUNA] Best PR-AUC for Random Forest: 0.8064
[OPTUNA] Best params: {'n_estimators': 100, 'max_depth': 12, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 0.5}
[OPTUNA] Running 10 trials for XGBoost ...
[OPTUNA] Best PR-AUC for XGBoost: 0.8091
[OPTUNA] Best params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.029311198685320016, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8059264473611898, 'reg_alpha': 0.0004982752357076451, 'reg_lambda': 0.0028888383623653178, 'min_child_weight': 4}


In [ ]:
# CELL 5 — Train Final Models on Full Training Set & Save

rf_model = train_final_model(
    model_name="rf",
    best_params=rf_study.best_params,
    X_train=X_train,
    y_train=y_train,
)

xgb_model = train_final_model(
    model_name="xgb",
    best_params=xgb_study.best_params,
    X_train=X_train,
    y_train=y_train,
    scale_pos_weight=scale_pos_weight,
)

save_models(rf_model, xgb_model, RF_PATH, XGB_PATH)

[TRAIN] rf final fit complete.
[TRAIN] xgb final fit complete.
[SAVE] Random Forest saved to /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/random_forest_baseline.joblib
[SAVE] XGBoost saved to /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/xgboost_champion.joblib


In [ ]:
# CELL 6 — Model Evaluation on OOT "Crypto Winter" Holdout

rf_metrics = evaluate_model("Random Forest Baseline", rf_model, X_test, y_test)
xgb_metrics = evaluate_model("XGBoost Champion", xgb_model, X_test, y_test)

EVALUATION REPORT — Random Forest Baseline (Out-of-Time Test Set)
Metric            | Score
------------------+----------
ROC-AUC           | 0.9183
PR-AUC            | 0.8490
F1-Score          | 0.7411
Recall            | 0.7948
Precision         | 0.6943

--- Confusion Matrix ---
Predicted:      0       1
Actual 0:  25630    3779
Actual 1:   2216    8582

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.87      0.90     29409
           1       0.69      0.79      0.74     10798

    accuracy                           0.85     40207
   macro avg       0.81      0.83      0.82     40207
weighted avg       0.86      0.85      0.85     40207

EVALUATION REPORT — XGBoost Champion (Out-of-Time Test Set)
Metric            | Score
------------------+----------
ROC-AUC           | 0.9200
PR-AUC            | 0.8520
F1-Score          | 0.7496
Recall            | 0.8064
Precision         | 0.7003

--- Confusion Matrix ---
Predic

In [ ]:
# CELL 7 — Export Artifacts

import numpy as np

# -- Prepare probability arrays for plotting --
rf_y_prob  = rf_model.predict_proba(X_test)[:, 1]
xgb_y_prob = xgb_model.predict_proba(X_test)[:, 1]

models_data = [
    {"name": "RF Baseline",       "y_test": y_test, "y_prob": rf_y_prob},
    {"name": "XGBoost Champion",  "y_test": y_test, "y_prob": xgb_y_prob},
]

plot_evaluation_curves(models_data=models_data, output_path=CURVES_PATH)

# -- Retrieve feature names from parquet schema (via Polars) --
import polars as pl
feature_names = [
    c for c in pl.read_parquet(TRAIN_PATH, n_rows=1).columns if c != "target"
]

plot_shap_summary(
    model=xgb_model,
    X_test=X_test,
    feature_names=feature_names,
    output_path=SHAP_PATH,
    top_n=15,
    max_display_samples=2000,
)

comparison_df = build_comparison_matrix(
    rf_metrics=rf_metrics,
    xgb_metrics=xgb_metrics,
    rf_study=rf_study,
    xgb_study=xgb_study,
    output_path=MATRIX_PATH,
)

[PLOT] Evaluation curves saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/model_evaluation_curves.png
[SHAP] SHAP summary plot saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/shap_summary_plot.png
MODEL COMPARISON MATRIX
Model                  | ROC-AUC | PR-AUC  | F1     | Recall | Precision | CV PR-AUC | Champion
-----------------------+---------+---------+--------+--------+-----------+-----------+---------
Random Forest Baseline | 0.9183  | 0.8490  | 0.7411 | 0.7948 | 0.6943    | 0.8064    | No      
XGBoost Champion       | 0.9200  | 0.8520  | 0.7496 | 0.8064 | 0.7003    | 0.8091    | YES     
[EXPORT] Comparison matrix saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/model_comparison_matrix.csv


In [ ]:
# CELL 8 — Final Model Selection & Justification

print_champion_justification(rf_metrics=rf_metrics, xgb_metrics=xgb_metrics)

FINAL MODEL SELECTION & JUSTIFICATION
Champion Model : XGBoost (Gradient Boosting)

[1] PR-AUC SUPERIORITY (Primary Criterion)
    XGBoost PR-AUC : 0.8520
    RF      PR-AUC : 0.8490
    Delta          : +0.0029  --> XGBoost wins on imbalanced precision-recall space.

[2] RECALL ON DEFAULT CLASS (Business Critical)
    XGBoost Recall : 0.8064
    RF      Recall : 0.7948
    Delta          : +0.0116  --> Higher recall = fewer missed defaults = lower credit loss.

[3] TEMPORAL ROBUSTNESS (Out-of-Time Crypto Winter)
    Both models evaluated exclusively on OOT holdout to prevent data leakage.
    XGBoost generalises better under distribution shift (Crypto Winter stress).

[4] REGULARISATION & MULTICOLLINEARITY CONTROL
    reg_alpha (L1) and reg_lambda (L2) tuned by Optuna to suppress redundant
    market-feature correlations (e.g., correlated price/volume indicators).
    scale_pos_weight dynamically set to counteract 12%+ default rate imbalance.

[5] INTERPRETABILITY — SHAP
    TreeExpla